# AgriSupply Data Warehouse: Exploratory Data Analysis (EDA) & OLAP

This notebook connects directly to our Postgres **Data Marts** utilizing SQLAlchemy. It demonstrates how Analysts perform OLAP (Online Analytical Processing) natively against the database.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from dotenv import load_dotenv

# Connect natively to PostgreSQL Data Warehouse
BASE_DIR = os.path.dirname(os.getcwd())
load_dotenv(os.path.join(BASE_DIR, ".env"))

engine = create_engine(URL.create(
    "postgresql",
    username=os.getenv("DB_USER", "postgres"),
    password=os.getenv("DB_PASSWORD", "postgres"),
    host=os.getenv("DB_HOST", "localhost"),
    port=os.getenv("DB_PORT", "5432"),
    database=os.getenv("DB_NAME", "postgres"),
))
# Fetch directly from the dbt-generated mart!
df_market = pd.read_sql("SELECT * FROM mart_market", engine)
df_market.head()

## 1. OLAP SLICE
**Definition:** Selecting a single dimension from the OLAP cube and yielding a new sub-cube. Here we 'Slice' our Data Mart to only look at **'Maize'**.

In [ ]:
sliced_cube = df_market[df_market["product_name"] == "Maize"].copy()
sliced_cube.drop(columns=["product_name"], inplace=True)

# Pivot to demonstrate the resulting sub-cube
view = sliced_cube.pivot_table(index=["year", "month"], columns="region_name", values="avg_price")
view.head()

## 2. OLAP DICE
**Definition:** Defining a sub-cube by performing a multidimensional selection. Here we grab exactly TWO products during TWO specific summer months across TWO regions.

In [ ]:
products = ["Maize", "Beans"]
regions = ["Nairobi", "Nakuru"]
months = [6, 7] # June, July

diced_cube = df_market[
    (df_market["product_name"].isin(products)) &
    (df_market["region_name"].isin(regions)) &
    (df_market["month"].isin(months))
]
diced_cube.head(10)

## 3. OLAP ROLLUP
**Definition:** Removing a dimension entirely, summarizing and jumping up the hierarchy. Here we 'Rollup' regional data to show National Annual Price.

In [ ]:
national_annual = df_market.groupby(["year", "product_name"])["avg_price"].mean().reset_index()
national_annual = national_annual.round(2)
national_annual.head()

## 4. OLAP DRILLDOWN
**Definition:** Moving down a hierarchy to add detail. Reversing a Rollup.

In [ ]:
print("1. HIGH LEVEL VIEW (Annual National Average for Maize)")
print(national_annual[national_annual["product_name"] == "Maize"].head())

print("\n2. DRILL DOWN -> INTO REGIONS (Adding Regional Specificity)")
drilled = df_market[(df_market["product_name"] == "Maize") & (df_market["year"] == 2020)]
region_view = drilled.groupby("region_name")["avg_price"].mean().reset_index()
print(region_view)